In [1]:
import os
import json

import pandas as pd
import jsonlines
from openai import OpenAI
from pydantic import BaseModel
from tqdm.notebook import tqdm

DATA_PATH_EXTRACTED = '../topic_analysis_gravidez_results_two_steps_api/retained_questions_with_topics.csv'
DATA_PATH_ORIGINALS = '../data/cleaned/cleaned_comments_info.csv'
JSON_PATH = '../data/results/question_answer/question_answer_results.jsonl'
OUTPUT_PATH = '../data/results/question_answer/question_answer_results.csv'
SAMPLE_PATH = '../data/results/question_answer/question_answer_sample.csv'

MODEL_NAME = 'gpt-5.4-mini'
REASONING_EFFORT = 'medium'
SERVICE_TIER = 'flex'

client = OpenAI()

os.makedirs(os.path.dirname(SAMPLE_PATH), exist_ok=True)

In [2]:
class AnswerDetection(BaseModel):
    tem_resposta: bool

In [3]:
CLASSIFICATION_PROMPT = """
Sua tarefa é determinar se um comentário filho responde à pergunta extraída
de um comentário pai. Você receberá um JSON com os campos titulo,
comentario_pai, pergunta_extraida e comentario_filho.

Avalie apenas se existe uma tentativa de responder. Não avalie a correção
factual: uma resposta errada, imprecisa ou parcial ainda pode ser classificada
como true.

Classifique tem_resposta como:

- true: o comentário filho responde, esclarece, aconselha ou fornece informação
  que ajuda diretamente a resolver a pergunta extraída. Relatos pessoais contam
  apenas quando são usados para responder ou orientar sobre a dúvida.

- false: o comentário filho não ajuda a resolver a pergunta. Isso inclui apenas
  relatar uma experiência sobre o mesmo tema, repetir a dúvida, fazer outra
  pergunta, dizer que possui a mesma dúvida, reagir, agradecer, mudar de assunto
  ou produzir um texto incompreensível.

Relevância temática não é suficiente. O comentário precisa apresentar alguma
informação, explicação, orientação ou conclusão relacionada à pergunta.

Use titulo e comentario_pai somente para compreender o contexto. Baseie a
classificação principalmente na relação entre pergunta_extraida e
comentario_filho.
"""

In [4]:
df_extracted = pd.read_csv(DATA_PATH_EXTRACTED)
df_originals = pd.read_csv(DATA_PATH_ORIGINALS)

In [5]:
# Uma linha por pergunta extraída
df_perguntas = (
    df_extracted[
        ['comment_id', 'titulo', 'comentario', 'perguntas']
    ]
    .reset_index()
    .rename(columns={
        'index': 'pergunta_id',
        'comment_id': 'comentario_pai_id',
        'perguntas': 'pergunta_extraida',
        'comentario': 'comentario_pai'
    })
)

# Uma linha por comentário filho
df_filhos = (
    df_originals
    .dropna(subset=['parent_id'])
    .loc[lambda df: df['parent_id'].ne('')]
    .rename(columns={
        'comment_id': 'comentario_filho_id',
        'parent_id': 'comentario_pai_id',
        'comment': 'comentario_filho'
    }) # pyright: ignore[reportCallIssue]
)

# Produto pergunta × filhos do respectivo comentário
df_perguntas_filhos = df_perguntas.merge(
    df_filhos[
        [
            'comentario_pai_id',
            'comentario_filho_id',
            'comentario_filho',
        ]
    ],
    on='comentario_pai_id',
    how='inner',
    validate='many_to_many',
)
df_perguntas_filhos = df_perguntas_filhos[
    [
        'pergunta_id',
        'titulo',
        'comentario_pai_id',
        'comentario_pai',
        'pergunta_extraida',
        'comentario_filho_id',
        'comentario_filho'
    ]
]
df_perguntas_filhos.shape

(5263, 7)

In [6]:
sample_parent_ids = df_perguntas_filhos['comentario_pai_id'].drop_duplicates().sample(n=150, random_state=42)
df_sample = df_perguntas_filhos[df_perguntas_filhos['comentario_pai_id'].isin(sample_parent_ids)].reset_index(drop=True)
df_sample.to_csv(SAMPLE_PATH, index=False)

In [7]:
def classify_comments(
    titulo: str,
    comentario_pai: str,
    pergunta_extraida: str,
    comentario_filho: str,
) -> AnswerDetection:

    user_content = json.dumps({
        "titulo": titulo,
        "comentario_pai": comentario_pai,
        "pergunta_extraida": pergunta_extraida,
        "comentario_filho": comentario_filho
    }, ensure_ascii=False)

    response = client.responses.parse(
        model=MODEL_NAME,
        instructions=CLASSIFICATION_PROMPT,
        input=user_content,
        reasoning={"effort": REASONING_EFFORT},
        text_format=AnswerDetection,
        service_tier=SERVICE_TIER
    )

    if response.output_parsed is None:
        raise ValueError('Não foi possível analisar a resposta do modelo.')

    return response.output_parsed

In [8]:
def extract_and_save(
    pergunta_id: int,
    titulo: str,
    comentario_pai_id: str,
    comentario_pai: str,
    pergunta_extraida: str,
    comentario_filho: str,
    comentario_filho_id: str,
    output_path: str=JSON_PATH
):
    result = classify_comments(
        titulo=titulo,
        comentario_pai=comentario_pai,
        pergunta_extraida=pergunta_extraida,
        comentario_filho=comentario_filho,
    )

    output = {
        'pergunta_id': pergunta_id,
        'titulo': titulo,
        'comentario_pai_id': comentario_pai_id,
        'comentario_pai': comentario_pai,
        'pergunta_extraida': pergunta_extraida,
        'comentario_filho_id': comentario_filho_id,
        'comentario_filho': comentario_filho,
        'tem_resposta': result.tem_resposta
    }

    with jsonlines.open(output_path, 'a') as writer:
        writer.write(output)

In [9]:
processed = set()
try:
    with jsonlines.open(JSON_PATH, 'r') as reader:
        processed = {
            (row['pergunta_id'], row['comentario_filho_id'])
            for row in reader
        }
except FileNotFoundError:
    pass

print(len(processed))

190


In [10]:
df_perguntas_filhos = df_sample.copy()

In [11]:
for _, row in tqdm(df_perguntas_filhos.iterrows(), total=df_perguntas_filhos.shape[0], desc="Processing..."):
    pergunta_id = row['pergunta_id']
    titulo = row['titulo']
    comentario_pai_id = row['comentario_pai_id']
    comentario_pai = row['comentario_pai']
    pergunta_extraida = row['pergunta_extraida']
    comentario_filho_id = row['comentario_filho_id']
    comentario_filho = row['comentario_filho']

    chave = (pergunta_id, comentario_filho_id)

    # Se este par já foi processado, pula para o próximo.
    if chave in processed:
        continue

    extract_and_save(
        pergunta_id=pergunta_id,
        titulo=titulo,
        comentario_pai_id=comentario_pai_id,
        comentario_pai=comentario_pai,
        pergunta_extraida=pergunta_extraida,
        comentario_filho_id=comentario_filho_id,
        comentario_filho=comentario_filho
    )
    processed.add(chave)

Processing...:   0%|          | 0/324 [00:00<?, ?it/s]

In [12]:
df_results = pd.read_json(JSON_PATH, lines=True)
df_results.to_csv(OUTPUT_PATH, index=False)

df_results['tem_resposta'].value_counts(dropna=False)

tem_resposta
False    167
True     157
Name: count, dtype: int64